# Store shelves web: Stage-2C vs residual five-stream Transformer

Evaluate-only notebook on the 130-image web holdout. Ground-truth target coordinates are never used for detection, cropping or ranking.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, zipfile

INPUT = Path('/kaggle/input')
PROJECT = Path('/kaggle/working/VIscaNEr_store_shelves_eval')
OUTPUT = Path('/kaggle/working/store_shelves_web_evaluation')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

def input_root(slug):
    matches = sorted({p.resolve() for pattern in (slug, f'*/{slug}', f'*/*/{slug}') for p in INPUT.glob(pattern) if p.is_dir()})
    if len(matches) != 1:
        raise RuntimeError(f'Attach exactly one {slug}; found: {matches}')
    return matches[0]

def unique_file(root, filename):
    matches = list(root.rglob(filename))
    if len(matches) != 1:
        raise RuntimeError(f'Expected one {filename} under {root}; found: {matches}')
    return matches[0]

print('SETUP | resolving holdout, latest Stage-2C and latest residual Transformer...', flush=True)
code = input_root('viscaner-store-shelves-web-eval-code')
holdout = input_root('viscaner-store-shelves-web')
stage2c_root = input_root('viscaner-stage2c-checkpoint')
easyocr_weights = input_root('viscaner-easyocr-ru-en-weights')
labels = input_root('viscaner-dinov3-data')
bottles = input_root('viscaner-bottle-classifier-data')
stage2c_checkpoint = unique_file(stage2c_root, 'manual_stage2c_best.pt')
residual_checkpoint = unique_file(INPUT, 'residual_best.pt')
joint_yolo = unique_file(code, 'best_joint_yolo.pt')

shutil.rmtree(PROJECT, ignore_errors=True)
shutil.copytree(code, PROJECT, ignore=shutil.ignore_patterns('dataset-metadata.json'))
sys.path.insert(0, str(PROJECT))
from fusion_stage2.kaggle_io import materialize_dataset_dirs
cache = Path('/kaggle/working/dataset_cache')
labels = materialize_dataset_dirs(labels, ('refs',), cache)
bottles = materialize_dataset_dirs(bottles, ('refs',), cache)
for required in (holdout/'labels.csv', holdout/'queries', labels/'refs', bottles/'refs', stage2c_checkpoint, residual_checkpoint, joint_yolo, easyocr_weights/'craft_mlt_25k.pth', easyocr_weights/'cyrillic_g2.pth'):
    if not required.exists():
        raise FileNotFoundError(required)
print('SETUP | inputs resolved; intentionally skipping multi-GB checkpoint hashing', flush=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=5.6,<6', 'safetensors>=0.5,<1', 'Unidecode>=1.3,<2', 'easyocr==1.7.2', 'ultralytics>=8.3,<9'], check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU accelerator before running this notebook')
print('SETUP | GPU:', torch.cuda.get_device_name(0), '| evaluation rows=130', flush=True)
OUTPUT.mkdir(parents=True, exist_ok=True)
command = [
    sys.executable, '-u', '-m', 'five_stream_transformer.evaluate_store_shelves_web',
    '--dataset-root', str(holdout), '--label-data-root', str(labels),
    '--bottle-data-root', str(bottles),
    '--stage2c-checkpoint', str(stage2c_checkpoint),
    '--residual-checkpoint', str(residual_checkpoint),
    '--joint-yolo', str(joint_yolo), '--easyocr-model-dir', str(easyocr_weights),
    '--output-dir', str(OUTPUT), '--device', 'cuda',
    '--feature-batch-size', '48', '--num-workers', '2',
]
print('EVAL COMMAND |', ' '.join(command), flush=True)
subprocess.run(command, cwd=PROJECT, check=True)
report = json.loads((OUTPUT/'evaluation.json').read_text())
print('NOTEBOOK COMPLETE |', json.dumps(report['results'], ensure_ascii=False), flush=True)